# Biomedical NER Evaluation on MACCROBAT

This notebook evaluates `d4data/biomedical-ner-all` against all 400 documents in `ktgiahieu/maccrobat2018_2020`.

Because the pretrained model may already have seen MACCROBAT during fine-tuning, these results are a full-dataset benchmark rather than an independent test of generalization.

## Load the Model

In [2]:
from __future__ import annotations

import numpy as np
import torch
from datasets import load_dataset
from seqeval.metrics import accuracy_score, f1_score, precision_score, recall_score
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)

MODEL_NAME = "d4data/biomedical-ner-all"
IGNORE_LABEL_ID = -100

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME)

print(f"Loaded {MODEL_NAME}.")
print("Evaluation device:", "cuda" if torch.cuda.is_available() else "cpu")
print("Model labels:", len(model.config.label2id))

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Loaded d4data/biomedical-ner-all.
Evaluation device: cpu
Model labels: 84


## Load MACCROBAT

In [3]:
maccrobat_evaluation = load_dataset(
    "ktgiahieu/maccrobat2018_2020",
    split="train",
)

print("Evaluation documents:", len(maccrobat_evaluation))
print("Dataset columns:", maccrobat_evaluation.column_names)

Evaluation documents: 400
Dataset columns: ['tokens', 'tags']


## Check Label Compatibility

Every MACCROBAT label must exist in the model label mapping. Extra model labels are allowed; predictions using them will count as errors.

In [4]:
dataset_labels = {
    tag
    for document_tags in maccrobat_evaluation["tags"]
    for tag in document_tags
}
model_labels = set(model.config.label2id)

missing_from_model = sorted(dataset_labels - model_labels)
extra_in_model = sorted(model_labels - dataset_labels)

print("MACCROBAT labels:", len(dataset_labels))
print("Model labels:", len(model_labels))
print("Missing from model:", missing_from_model)
print("Extra in model:", extra_in_model)

if missing_from_model:
    raise ValueError("The model does not support every MACCROBAT label.")

MACCROBAT labels: 82
Model labels: 84
Missing from model: []
Extra in model: ['B-Non[biological](Detailed_description', 'B-Personal_[back](Biological_structure']


## Tokenize and Align BIO Labels

Each word label is assigned to its first subtoken. Special tokens and remaining subtokens receive `-100` so they are ignored by the metrics. Long documents are divided into non-overlapping 512-token chunks.

In [5]:
def tokenize_and_align_maccrobat(examples):
    tokenized = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=512,
        stride=0,
        return_overflowing_tokens=True,
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    aligned_labels = []

    for chunk_index, sample_index in enumerate(sample_mapping):
        word_ids = tokenized.word_ids(batch_index=chunk_index)
        document_tags = examples["tags"][sample_index]
        chunk_labels = []
        previous_word_id = None

        for word_id in word_ids:
            if word_id is None or word_id == previous_word_id:
                chunk_labels.append(IGNORE_LABEL_ID)
            else:
                chunk_labels.append(model.config.label2id[document_tags[word_id]])
            previous_word_id = word_id

        aligned_labels.append(chunk_labels)

    tokenized["labels"] = aligned_labels
    return tokenized


tokenized_evaluation = maccrobat_evaluation.map(
    tokenize_and_align_maccrobat,
    batched=True,
    remove_columns=maccrobat_evaluation.column_names,
    desc="Tokenizing full MACCROBAT evaluation set",
)

print("Evaluation documents:", len(maccrobat_evaluation))
print("Tokenized chunks:", len(tokenized_evaluation))
print("Tokenized columns:", tokenized_evaluation.column_names)

Evaluation documents: 400
Tokenized chunks: 730
Tokenized columns: ['input_ids', 'attention_mask', 'labels']


## Calculate Metrics

Precision, recall, and F1 are entity-level BIO metrics from `seqeval`. Accuracy is token-level and includes the outside label `O`.

In [6]:
def compute_ner_metrics(evaluation_result):
    logits, label_ids = evaluation_result
    predicted_ids = np.argmax(logits, axis=-1)
    predictions = []
    references = []

    for predicted_row, label_row in zip(predicted_ids, label_ids):
        kept_predictions = []
        kept_references = []

        for predicted_id, label_id in zip(predicted_row, label_row):
            if label_id == IGNORE_LABEL_ID:
                continue
            kept_predictions.append(model.config.id2label[int(predicted_id)])
            kept_references.append(model.config.id2label[int(label_id)])

        predictions.append(kept_predictions)
        references.append(kept_references)

    return {
        "precision": precision_score(references, predictions, zero_division=0),
        "recall": recall_score(references, predictions, zero_division=0),
        "f1": f1_score(references, predictions, zero_division=0),
        "accuracy": accuracy_score(references, predictions),
    }

In [7]:
evaluation_arguments = TrainingArguments(
    output_dir="outputs/biomedical-ner-maccrobat-evaluation",
    per_device_eval_batch_size=8,
    report_to="none",
)

evaluator = Trainer(
    model=model,
    args=evaluation_arguments,
    eval_dataset=tokenized_evaluation,
    processing_class=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_ner_metrics,
)

evaluation_metrics = evaluator.evaluate()
evaluation_metrics

/opt/anaconda3/envs/dl-lab/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Step,Precision,Recall,F1,Accuracy
No log,2.315807,0,0.340538,0.509077,0.408091,0.741173


{'eval_loss': 2.315807342529297,
 'eval_precision': 0.34053770426265556,
 'eval_recall': 0.5090766472433886,
 'eval_f1': 0.40809054705512476,
 'eval_accuracy': 0.7411729358287216}